# Visual validation against the Weisenbach IPM implementation

This optional integration executes or loads the independent Weisenbach/Luke
implementation and compares it with `microcaustics` on exactly the same square
source aperture.  The adapter rotates the shear and stellar catalog into the
external $y_1/y_2$ convention before execution, then rotates the native map
back.  It does not shoot a larger field and crop the corners.

The first result is a registered three-panel map comparison.  The second is a
ten-year point-source light curve generated directly from every map in both
sequences.  Set `MICROCAUSTICS_RUN_WEISENBACH=1` to rerun the external shell
script. Otherwise the notebook reuses its independently generated checkpoints.

In [ ]:
import json
import os
import subprocess
from pathlib import Path
import numpy as np
import torch
from matplotlib import pyplot as plt
import microcaustics as mc
import microcaustics.plotting as mcp

plt.rcParams.update(mcp.publication_style())

def workspace_root():
    candidates = (Path(mcp.__file__).resolve().parents[4], Path.cwd(), *Path.cwd().parents)
    return next(
        (path for path in candidates if (path / "microcaustics_package").is_dir() and (path / "output").is_dir()),
        Path.cwd(),
    )

workspace = workspace_root()
default_problem = workspace / "output/final_paper_run_production_v4/quasar/Q2237+0305_image_B/results_paper_benchmark"
problem_root = Path(os.environ.get("MICROCAUSTICS_WEISENBACH_ROOT", default_problem))
luke_root = problem_root / "luke_external"
runner = problem_root / "run_luke_light_curve.sh"
if os.environ.get("MICROCAUSTICS_RUN_WEISENBACH") == "1":
    if os.name == "nt":
        subprocess.run(["wsl.exe", "bash", str(runner).replace("C:\\", "/mnt/c/").replace("\\", "/")], check=True)
    else:
        subprocess.run(["bash", str(runner)], check=True)

aggregate_path = Path(os.environ.get(
    "MICROCAUSTICS_WEISENBACH_NPZ",
    luke_root / "luke_lightcurve_all_frames.npz",
))
frame_index = 29
frame_path = Path(os.environ.get(
    "MICROCAUSTICS_WEISENBACH_FRAME_NPZ",
    luke_root / f"luke_frame_{frame_index:04d}.npz",
))
problem_json = problem_root / "paper_external_problem_definition.json"
star_field_path = problem_root / "paper_external_problem_star_fields.npz"
required = (aggregate_path, frame_path, problem_json, star_field_path)
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing external validation products. Set MICROCAUSTICS_WEISENBACH_ROOT "
        "or run the external adapter first:\n" + "\n".join(missing)
    )
print("External products:", luke_root)
print(mcp.runtime_description())

## Reconstruct the registered `microcaustics` problem

The JSON and NPZ files are a solver-independent benchmark contract.  The
stellar positions at all 147 epochs are linear here, so the public
`PointMassField` is reconstructed from frame zero and the measured velocities.
The production $N=10^7$, $k=2$, $r=2$, $v=4$ tiled IPM settings then generate
our maps without reading a saved `microcaustics` light curve.

In [ ]:
problem = json.loads(problem_json.read_text())
with np.load(star_field_path, allow_pickle=False) as data:
    times = torch.from_numpy(data["time_days"].copy()).to(torch.float32)
    star_x = torch.from_numpy(data["star_x_uas"].copy())
    star_y = torch.from_numpy(data["star_y_uas"].copy())
    mass_solar = torch.from_numpy(data["m_stars_msun"].copy())
    catalog_theta_e = torch.from_numpy(data["theta_ein_star_uas"].copy())
# Weisenbach's solver uses an axis-aligned shear eigenframe.  Rotate the
# complete physical problem into that frame before either solver runs; this
# preserves the full square aperture without padding or extra rays.
macro_values = problem["macro_lens"]
# Match the external benchmark cosmology, not the package's default cosmology.
distances = mc.LensingDistances.from_redshifts(
    macro_values["z_lens"], macro_values["z_source"], H0=70.0, Om0=0.3,
)
if not torch.allclose(
    distances.einstein_radius_uas(mass_solar).cpu(), catalog_theta_e,
    rtol=2e-5, atol=1e-6,
):
    raise ValueError("The benchmark cosmology and imported Einstein radii do not match")
phi = float(macro_values["shear_phi_rad"])
cos_phi, sin_phi = np.cos(phi), np.sin(phi)
native_x = -sin_phi * star_x + cos_phi * star_y
native_y = -cos_phi * star_x - sin_phi * star_y
duration = float(times[-1] - times[0])
velocity_x = (native_x[-1] - native_x[0]) / duration
velocity_y = (native_y[-1] - native_y[0]) / duration
field = mc.PointMassField(
    x_uas=native_x[0], y_uas=native_y[0], mass_solar=mass_solar,
    velocity_x_uas_per_day=velocity_x, velocity_y_uas_per_day=velocity_y,
)
macro = mc.MacroLens(
    convergence=macro_values["kappa_tot"],
    shear=macro_values["gamma_tot"],
    shear_angle_deg=-90.0,
    smooth_matter_fraction=macro_values["smooth_matter_fraction_sm"],
)
capabilities = mc.RuntimeCapabilities.detect()
use_cuda = capabilities.cuda_available
runtime_config = mc.RuntimeConfig(
    device="cuda" if use_cuda else "cpu",
    backend="triton" if use_cuda and capabilities.triton_importable else "auto",
    dtype=torch.float32,
)
lens_width = float(problem["lens_plane"]["star_field_len_scale_uas"])
source_width = float(problem["source_grid"]["sp_fov_uas"])
source_resolution = int(problem["source_grid"]["Nsrc"])
lens_region = mc.PlaneRegion((lens_width, lens_width))
source_grid = mc.PlaneGrid((source_resolution, source_resolution), (source_width, source_width))
system = mc.MicrolensingSystem(
    macro=macro, distances=distances,
    stars=field, source_grid=source_grid, lens_region=lens_region,
    duration_days=duration, runtime=runtime_config,
)
simulation = system.realize().simulation  # resolved tracer for the external solver audit
method = mc.production_ipm_config(
    rays=10_000_000 if use_cuda else 262_144,
)
schedule = mc.production_dynamic_config()

our_mu = []
our_frame = None
for index, product in enumerate(simulation.dynamic_maps(
    lens_region, source_grid, times.tolist(), method=method, schedule=schedule
)):
    center = product.values[source_resolution // 2 - 1:source_resolution // 2 + 2,
                            source_resolution // 2 - 1:source_resolution // 2 + 2]
    our_mu.append(center.mean().detach().cpu())
    if index == frame_index:
        our_frame = mc.MagnificationMap(
            product.values.detach().cpu(), product.grid,
            time_days=product.time_days, method="microcaustics IPM",
            metadata=product.metadata,
        )
our_mu = torch.stack(our_mu)
assert our_frame is not None


## Registered maps and residual

Both panels cover the identical aperture at 100% finite coverage.  The
residual is $-2.5\log_{10}(\mu_{\mathrm{microcaustics}}/\mu_{\mathrm{Weisenbach}})$.
there is no rotation, padding, or post-hoc crop in this comparison.

In [ ]:
with np.load(frame_path, allow_pickle=False) as data:
    # Luke stores [row=y2 descending, column=y1]; PlaneGrid arrays use
    # [row=y ascending, column=x], so only the row direction changes.
    luke_values = torch.from_numpy(data["mag_map_shear_aligned"][::-1, :].copy())
    coverage = float(torch.isfinite(luke_values).to(torch.float64).mean())
    if coverage != 1.0:
        raise RuntimeError(f"Registered native-map coverage is {coverage:.3%}, not 100%")
    extent = tuple(float(value) for value in data["raw_extent_shear_uas"])
luke_grid = mc.PlaneGrid(
    luke_values.shape,
    (extent[3] - extent[2], extent[1] - extent[0]),
    ((extent[2] + extent[3]) / 2, (extent[0] + extent[1]) / 2),
)
fov_error = max(abs(a - b) for a, b in zip(luke_grid.field_of_view_uas, our_frame.grid.field_of_view_uas))
center_error = max(abs(a - b) for a, b in zip(luke_grid.center_uas, our_frame.grid.center_uas))
pixel_tolerance = 0.01 * min(our_frame.grid.pixel_scale_uas)
if fov_error > pixel_tolerance or center_error > pixel_tolerance:
    raise RuntimeError("External and microcaustics maps do not share one registered aperture")
# The external float32 extent differs from the JSON contract only by rounding;
# attach the common contract grid without interpolation or cropping.
luke_frame = mc.MagnificationMap(
    luke_values,
    our_frame.grid,
    time_days=float(times[frame_index]),
    method="Weisenbach IPM",
)
figure, axes = mcp.plot_map_comparison(
    our_frame,
    luke_frame,
    residual="magnitude",
    residual_limit=0.12,
    scale_bar_uas=1.0,
    show_axes=False,
)
axes[0].set_title("microcaustics IPM")
axes[1].set_title("Weisenbach IPM")
axes[2].set_title("Residual")
figure.tight_layout()
print("Registered aperture coverage:", f"{coverage:.1%}")

## Independently generated ten-year light curves

For an immediately reproducible solver comparison, both curves sample the
same central $3	imes3$ pixels of every source-plane map.  An explicit 20-mag
unlensed baseline converts absolute magnification to plotted brightness.
Finite-source comparisons use the same generated maps and replace this small
aperture with any desired source profile.

In [ ]:
def host_path(value):
    text = str(value)
    if os.name == "nt" and text.startswith("/mnt/c/"):
        return Path("C:/" + text[len("/mnt/c/"):])
    return Path(text)

with np.load(aggregate_path, allow_pickle=False) as aggregate:
    frame_paths = [host_path(value) for value in aggregate["frame_npz_paths"]]
if len(frame_paths) != len(times):
    raise RuntimeError("External frame-path manifest does not match the time grid")
luke_mu = []
for path in frame_paths:
    with np.load(path, allow_pickle=False) as data:
        values = torch.from_numpy(data["mag_map_shear_aligned"][::-1, :].copy())
    center = values[source_resolution // 2 - 1:source_resolution // 2 + 2,
                    source_resolution // 2 - 1:source_resolution // 2 + 2]
    luke_mu.append(center.mean())
luke_mu = torch.stack(luke_mu)
baseline_mag = 20.0
ours_mag = baseline_mag - 2.5 * torch.log10(our_mu.clamp_min(1.0e-12))
luke_mag = baseline_mag - 2.5 * torch.log10(luke_mu.clamp_min(1.0e-12))
residual_mmag = 1000.0 * (ours_mag - luke_mag)

figure, axes = plt.subplots(2, 1, figsize=(8.3, 5.8), sharex=True,
                            gridspec_kw={"height_ratios": (2.2, 1.0)})
axes[0].plot(times, ours_mag, color="#1f77b4", label="microcaustics IPM")
axes[0].plot(times, luke_mag, color="#d62728", label="Weisenbach IPM", alpha=0.85)
axes[0].invert_yaxis()
axes[0].set_ylabel("brightness [mag]")
axes[0].legend(frameon=True)
axes[1].plot(times, residual_mmag, color="black", linewidth=0.9)
axes[1].axhline(0.0, color="0.5", linewidth=0.7)
axes[1].set(xlabel="Observer time [days]", ylabel="Residual [mmag]")
for axis in axes:
    mcp.finish_axis(axis)
figure.tight_layout()